# 02 — Marin övervakning: Hunnebostrand, Sotenäs (Bohuslän)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_030-Marine-Vessels-Sotenas.ipynb)

**Syfte:** Demonstrera automatisk detektion av fartyg från Sentinel-2-bilder längs svenska västkusten (Hunnebostrand/Sotenäs). Användningsområden: sjöfartsövervakning, illegal fiske, havsförorening, försvarsunderrättelse.

**Partners som bidragit:**
- **SSC (Swedish Space Corporation)** — data, markstationer, operativ erfarenhet
- **Sjöfartsverket** — sjökort (S-57 via SLU GET)
- **PandionAI** — onboard AI-detektion (AlertSat-konstellation)
- **AI Sweden / RISE** — modellutveckling

**Datakällor:**
- Copernicus Sentinel-2 L2A via Digital Earth Sweden
- AI2 (Allen Institute) vessel detection model
- Fine-tuned YOLO11s för maritima objekt
- Sjökort S-57 (Sjöfartsverket, akademisk användning)

**Licens:** CC0 1.0 (notebook) · YOLO AGPL-3.0 · S-57 academic-only

## ⚠️ Licens — YOLO11s är AGPL-3.0

Denna notebook använder `marine_vessels`-analyzern som bygger på **YOLO11s** (Ultralytics, AGPL-3.0).

**Vad det innebär:**

| Användning | OK utan Enterprise License? |
|-----------|------------------------------|
| Binder / publik Jupyter (denna notebook) | ✅ Ja — vårt repo är publikt, AGPL-nätverksklausulen uppfylld |
| Lokal körning för forskning/utbildning | ✅ Ja — ingen distribution |
| DES JupyterHub (publik plattform) | ✅ Ja — open-source infrastruktur |
| Kommersiell sluten produkt eller betaltjänst | ❌ **Nej** — kräver [Ultralytics Enterprise License](https://www.ultralytics.com/license) |

**Alternativ utan AGPL:** `ai2_vessels` (Allen Institute-modell) fungerar som ersättare med annan licensprofil. `object_detection` i heatmap-läge är helt modellfri (ren variansmatematik) och kan användas för first-pass-screening utan licensbörda.

Se [ImintEngine THIRD_PARTY_LICENSES.md](https://github.com/TobiasEdman/imintengine/blob/main/THIRD_PARTY_LICENSES.md) för fullständig licensöversikt.

## 1. Metod

Två modeller, en heatmap, en landmaskning:

| Analyzer | Modell | Output |
|----------|--------|--------|
| `marine_vessels` | YOLO11s (fine-tuned) | Bounding boxes + confidence |
| `ai2_vessels` | AI2 vessel detector | Bounding boxes + klass |
| `object_detection` | Variance heatmap (ingen modell) | Anomaliregioner |
| `spectral` | NDWI | Land/vatten-mask |

**Pipeline:**

```
Sentinel-2 → NDWI-mask (filtrera land bort)
          → YOLO11s + AI2 (parallellt)
          → Överlapp-filter (non-max suppression)
          → Konfidenströskel (default 0.25)
```

## 2. Setup

In [ ]:
# Verified API helper — wraps IMINTResult access
def get_outputs(result, name):
    """Return outputs dict for first successful analyzer matching `name`, else None."""
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None


In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
import json

# Hunnebostrand, Sotenäs kommun, Bohuslän
AOI = {
    "west": 11.30,
    "south": 58.40,
    "east": 11.55,
    "north": 58.55,
}
DATE = "2024-07-15"

print(f"AOI (Sotenäs): {AOI}")
print(f"Datum: {DATE}")

## 3. Kör analys

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/marin_sotenas",
    config_path="config/analyzers.yaml",
)

result = executor.execute(
    date=DATE,
    coords=AOI,
)

# Summera fartygsdetektioner
vessel_count = 0
if get_outputs(result, "marine_vessels") is not None:
    vessels = get_outputs(result, "marine_vessels").data.get("detections", [])
    vessel_count = len(vessels)
    print(f"YOLO11s fine-tuned: {vessel_count} fartyg")

if "ai2_vessels" in result.analyses:
    ai2 = get_outputs(result, "ai2_vessels").data.get("detections", [])
    print(f"AI2 vessel detector: {len(ai2)} fartyg")

## 4. Visualisering

In [ ]:
# Interaktiv karta med fartygsdetektioner som markörer
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Satellit",
).add_to(m)

folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#0066ff",
    weight=2,
    fill=False,
).add_to(m)

# Lägg till fartygsdetektioner om de finns
if get_outputs(result, "marine_vessels") is not None:
    for v in get_outputs(result, "marine_vessels").data.get("detections", []):
        if "lat" in v and "lon" in v:
            folium.CircleMarker(
                location=[v["lat"], v["lon"]],
                radius=4,
                color="red",
                fill=True,
                popup=f"Conf: {v.get('confidence', 0):.2f}",
            ).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Statiska paneler: RGB med bounding boxes + heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(job.rgb)
axes[0].set_title(f"Sentinel-2 RGB + YOLO detections ({DATE})")

if get_outputs(result, "marine_vessels") is not None:
    for v in get_outputs(result, "marine_vessels").data.get("detections", []):
        if "bbox" in v:
            x, y, w, h = v["bbox"]
            rect = mpatches.Rectangle(
                (x, y), w, h,
                linewidth=1.5, edgecolor="red", facecolor="none",
            )
            axes[0].add_patch(rect)
axes[0].axis("off")

if get_outputs(result, "object_detection") is not None:
    heatmap = get_outputs(result, "object_detection").data.get("heatmap")
    if heatmap is not None:
        axes[1].imshow(heatmap, cmap="hot")
        axes[1].set_title("Varians-heatmap (anomalier)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Tolkning

**Begränsningar med Sentinel-2 för fartygsdetektion:**
- 10 m upplösning → kan detektera fartyg > ~15 m
- 5-dagars återbesökstid → missar snabba händelser
- Molnfritt krävs → radar (Sentinel-1) kompletterar

**PandionAI:s värde:**
- Onboard AI på AlertSat-satelliter → realtidsdetektion utan ground-station-latency
- VHR-sensorer (sub-meter) → mindre fartyg kan detekteras

**Användningsfall:**
- Miljöövervakning (oljespill, bränsleutsläpp)
- Sjöfartsverkets AIS-komplettering (AIS kan stängas av — satellit fångar ändå)
- Kustbevakningen / Försvarets underrättelse
- Illegal fisketraling

**Nästa steg:**
- Fusion med AIS-data → identifiera "dark vessels" som inte sänder AIS
- Sentinel-1 SAR-data för molnfri coverage
- Tidserieanalys → normalt trafikmönster + avvikelser

### Länkar
- [ImintEngine marin-showcase](https://digitalearth.se/case/)
- [ImintEngine/imint/analyzers/marine_vessels.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/marine_vessels.py)
- [AI2 Satellite Vessel Detection](https://allenai.org/)
- [PandionAI AlertSat](https://www.pandionai.com/)